<a href="https://colab.research.google.com/github/RachelAlcraft/TrivialDivergence/blob/main/Triangles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
pip install ra-trivial

In [75]:
# Settings
import random
import math
import pandas as pd

#columns =['A', 'B','C','AB','BC','CA','CBp']
columns =['A', 'B','C']
#columns =['A', 'B','C','BC','AB','CA']
bins = 10
no_tri= 20
max_dims = len(columns)
dic_lists = {}

In [76]:
# triangle making function
def make_triangles(no_triangles,right_all_mix):

  # R=Right, A=any, M e half right half any for 2 states
  bond_length_angle_range = (45,55)
  bond_length_range_lower = (5,15)
  bond_length_range_mid = (5,15)
  bond_length_range_upper = (5,15)
  AB_range = [5,8]

  As = []
  Bs = []
  Cs = []
  ABs = []
  BCs = []
  CAs = []
  CBps = []

  for i in range(no_tri):
    this_r_a_m = right_all_mix
    if right_all_mix == "M":
      if i < no_tri/2:
        this_r_a_m = "R"
      else:
        this_r_a_m = "A"
    if this_r_a_m == "R":
      A = 90
    elif i == 0:
      A = 90 #I want to make sure I capture the random possibility of a different state so always have at least 1 right angle triangle
    else:
      A = random.randint(1, 178)
    B = random.randint(1, 180-A-1)
    C = 180 - (A+B)
    Ar = math.radians(A)
    Br = math.radians(B)
    Cr = math.radians(C)
    AB = random.randint(5,8)
    sin_ratio = AB / math.sin(Cr)
    BC = math.sin(Ar) * sin_ratio
    CA = math.sin(Br) * sin_ratio
    #CBp is the link to the next triangle
    if C < bond_length_angle_range[0]:
      CBp = random.randint(bond_length_range_lower[0],bond_length_range_lower[1])
    elif C > bond_length_angle_range[1]:
      CBp = random.randint(bond_length_range_upper[0],bond_length_range_upper[1])
    else:
      CBp = random.randint(bond_length_range_mid[0],bond_length_range_mid[1])
    #print(A,B,C,A+B+C,AB,BC,CA,CBp)
    As.append(A)
    Bs.append(B)
    Cs.append(C)
    ABs.append(AB)
    BCs.append(BC)
    CAs.append(CA)
    CBps.append(CBp)
  ####################################

  dic_lists["A"] = As
  dic_lists["B"] = Bs
  dic_lists["C"] = Cs
  dic_lists["AB"] = ABs
  dic_lists["BC"] = BCs
  dic_lists["CA"] = CAs
  dic_lists["CBp"] = CBps

  df_list = []
  for col in columns:
    df_list.append( dic_lists[col])

  df_transpose =  [list(i) for i in zip(*df_list)]
  df = pd.DataFrame(df_transpose,columns =columns)
  return df

In [77]:
# generate a triangle dataset

dfR = make_triangles(no_tri,"R")
dfA = make_triangles(no_tri,"A")
dfM = make_triangles(no_tri,"M")
#print(dfR)

dic_dfs_per_tri = {}
dic_dfs_per_tri["Mixed"] = dfM
dic_dfs_per_tri["Right"] = dfR
dic_dfs_per_tri["Any"] = dfA

#print(dfM)
#print(dfR)
#print(dfA)

print("Completed")

Completed


In [78]:
# Sampling each observation FUNCTION
# We now want to try again removing each obs and recalcing
def drop_row(df,cols,bins,dim,df_corr,show=False,only=False):
  list_x = df_corr["metric"].tolist()
  col_names = []
  for i in range(len(df_corr.index)):
    cols_nm = ""
    for c in range(dim):
      col_nm = f"col{c+1}"
      cols_nm += ":" + df_corr[col_nm][i]
    col_names.append(cols_nm[1:])
  if show:
    print(col_names)

  if show:
    print("x",list_x)
  # list of lists object for heatmap
  list_of_lists = []
  list_of_metrics = []
  for i in range(len(df.index)):
    dfx=df.drop(df.index[i])
    #if only: #instead of dropping we do it on its own
    #  dfx=df.index[i]
    rae_markx = awa.AlcraftWilliamsAssociation(dfx,bins=bins,piters=0)
    df_corrx = rae_markx.getStrongestAssociations([],cols,dim,1)
    list_i = df_corrx["metric"].tolist()
    df_corrx['is_it_bigger'] = df_corrx['metric'] > (df_corr['metric'])
    #df_corrx["is_it_bigger"] = df_corrx["is_it_bigger"].astype(str) #make it a string
    list_TF = df_corrx["is_it_bigger"].tolist()
    list_met = df_corrx["metric"].tolist()
    #print(i,list_i)
    if show:
      print(i,list_TF)
    list_of_lists.append(list_TF)
    list_of_metrics.append(list_met)
  return col_names,list_of_lists,list_of_metrics


In [79]:
# Create the trivial divergence
cols = columns
import ra_trivial.AlcraftWilliamsAssociation as awa

sets_corrs = {}

for tag,df in dic_dfs_per_tri.items():
#for tag,df in [("Mixed",dfM)]:
  rae_mark = awa.AlcraftWilliamsAssociation(df,bins=bins,piters=0)
  for i in range(2,max_dims+1):
    print(tag,"\tmake",i,"dims")
    df_corri = rae_mark.getStrongestAssociations([],cols,i,1)
    sets_corrs[f"{tag}_{i}"] = df_corri


print("Complete")

Mixed 	make 2 dims
Mixed 	make 3 dims
Right 	make 2 dims
Right 	make 3 dims
Any 	make 2 dims
Any 	make 3 dims
Complete


In [80]:
# make into a heatmap
import plotly.express as px
# https://plotly.com/python/heatmaps/

def make_heatmap(list_of_lists, col_names, title,hue):

  fig = px.imshow(list_of_lists,
                  labels=dict(x="Association", y="Observations?", color=hue),
                  x=col_names,
                  color_continuous_scale='Bluered_r',
                  title=title)
  fig.update_xaxes(side="top")
  fig.show()



In [54]:
# Create the plots and individual samples

for tag,df in dic_dfs_per_tri.items():
  print(tag)
  show_rows = False
  #only=True
  all_col_names = []
  dic_lists = {}
  dic_metrics = {}
  print("Drop rows")
  for i in range(2,max_dims+1):
    print(f"{i}/{max_dims}")
    dfi = sets_corrs[f"{tag}_{i}"]
    col_namesi,list_of_listsi,list_of_metricsi = drop_row(df,cols,bins,i,dfi,show=show_rows)
    dic_lists[i] = list_of_listsi
    dic_metrics[i] = list_of_metricsi
    all_col_names += col_namesi

  list_of_list_of_lists = []
  list_of_list_of_metrics = []
  num_obs = len(dic_lists[2]) # get the length of the first one, it is the number of observations
  print("Create obs lists")
  for ob in range(num_obs):
    lolol = []
    for key,lol in dic_lists.items():
      obs_lst = lol[ob]
      lolol += obs_lst
    list_of_list_of_lists.append(lolol)
  for ob in range(num_obs):
    lolol = []
    for key,lol in dic_metrics.items():
      obs_lst = lol[ob]
      lolol += obs_lst
    list_of_list_of_metrics.append(lolol)
  # Make HEATMAP
  print("number obs=",num_obs,tag)
  #make_heatmap(list_of_list_of_lists, all_col_names, title=f"{tag} Improves by removal?",hue="Improves?")
  make_heatmap(list_of_list_of_metrics, all_col_names,title=f"{tag} Metric with removal",hue="Triviality")

Mixed
Drop rows
2/3
3/3
Create obs lists
number obs= 6 Mixed


Right
Drop rows
2/3
3/3
Create obs lists
number obs= 6 Right


Any
Drop rows
2/3
3/3
Create obs lists
number obs= 6 Any


In [ ]:
# build a tree with all dims at the top and then each with a thing removed
# and the 10101 for each of those

In [83]:
# Function to calculate association and add it to a dictopnary
def calc_and_add_triv(df_obs,dic_all_metrics):
  #df_obs = dic_dfs_per_tri["Mixed"]
  rae_mark_obs = awa.AlcraftWilliamsAssociation(df_obs,bins=bins,piters=0)
  dic_obs = {}
  #dic_all_metrics = {}
  for i in range(2,max_dims+1):
    #print("make",i,"dims")
    df_corri = rae_mark.getStrongestAssociations([],cols,i,1)
    for idx in df_corri.index:
      triviality = df_corri["metric"][idx]
      triv_nm = ""
      for c in range(i):
        triv_nm += ":" + df_corri[f"col{c+1}"][idx]
      triv_nm = triv_nm[1:]
      if triv_nm not in dic_all_metrics:
        dic_all_metrics[triv_nm] = []
      dic_all_metrics[triv_nm].append(triviality)
      #if len(dic_all_metrics[triv_nm]) > 20:
      #  print(triv_nm)
  return dic_all_metrics

In [ ]:
# Split into all possible halves and retry
# First the original data
df_obs = dic_dfs_per_tri["Mixed"]
dic_all_metrics = {}
print("Calc all")
dic_all_metrics = calc_and_add_triv(df_obs,dic_all_metrics)
#rae_mark_obs = awa.AlcraftWilliamsAssociation(df_obs,bins=bins,piters=0)
#dic_obs = {}

#for i in range(2,max_dims+1):
#  print("make",i,"dims")
#  df_corri = rae_mark.getStrongestAssociations([],cols,i,1)
#  for idx in df_corri.index:
#    triviality = df_corri["metric"][idx]
#    triv_nm = ""
#    for c in range(i):
#      triv_nm += df_corri[f"col{c+1}"][idx] + ":"
#    triv_nm = triv_nm[1:]
#    if triv_nm not in dic_all_metrics:
#      dic_all_metrics[triv_nm] = []
#    dic_all_metrics[triv_nm].append(triviality)
#print(dic_all_metrics)

# Create the halves
import itertools
obs_list = range(0,len(df_obs.index))
half_obs = int(len(df_obs.index)/2)
print("There are",len(df_obs.index),"observations","half=",half_obs)

# Generate all possible two-element combinations
# Convert the resulting iterator to a list
combinations = list(itertools.combinations(obs_list, half_obs))
print("There are", len(combinations))
for i in range(len(combinations)):
  if i%100 == 0:
    print("Calc",i,"/",len(combinations))
  combo = combinations[i]
  combo_lst = []
  for cm in combo:
    combo_lst.append(cm)
  halved_df = df_obs.iloc[combo_lst]
  dic_all_metrics = calc_and_add_triv(halved_df,dic_all_metrics)

#print(dic_all_metrics)



Calc all
There are 20 observations half= 10
There are 184756
Calc 0 / 184756
Calc 100 / 184756
Calc 200 / 184756
Calc 300 / 184756
Calc 400 / 184756
Calc 500 / 184756
Calc 600 / 184756
Calc 700 / 184756
Calc 800 / 184756
Calc 900 / 184756
Calc 1000 / 184756
Calc 1100 / 184756
Calc 1200 / 184756
Calc 1300 / 184756
Calc 1400 / 184756
Calc 1500 / 184756
Calc 1600 / 184756
Calc 1700 / 184756
Calc 1800 / 184756
Calc 1900 / 184756
Calc 2000 / 184756
Calc 2100 / 184756
Calc 2200 / 184756
Calc 2300 / 184756
Calc 2400 / 184756
Calc 2500 / 184756
Calc 2600 / 184756
Calc 2700 / 184756
Calc 2800 / 184756
Calc 2900 / 184756
Calc 3000 / 184756
Calc 3100 / 184756
Calc 3200 / 184756
Calc 3300 / 184756
Calc 3400 / 184756
Calc 3500 / 184756
Calc 3600 / 184756
Calc 3700 / 184756
Calc 3800 / 184756
Calc 3900 / 184756
Calc 4000 / 184756
Calc 4100 / 184756
Calc 4200 / 184756
Calc 4300 / 184756
Calc 4400 / 184756
Calc 4500 / 184756
Calc 4600 / 184756
Calc 4700 / 184756
Calc 4800 / 184756
Calc 4900 / 184756
C

In [74]:
# make a heatmap frtom the samples
# turn dictonary into a list of cal names and a list of list
lol={}
col_header = []
for triv,vals in dic_all_metrics.items():
  col_header.append(triv)
  for v in range(len(vals)):
    if v not in lol:
      lol[v] = []
    lol[v].append(vals[v])
    #print(triv,v,vals[v])

lolol = []
for v,vals in lol.items():
  lolol.append(vals)
  #print(vals)
print(col_header)
make_heatmap(lolol, col_header,title=f"Triviality in halves",hue="Triviality")


['A:C', 'A:B', 'B:C', 'A:B:C']
